# 异步 IO

## 协程 Coroutine
> 线程——执行一个子程序

> 协程——在子程序执行过程中可以中断，去执行其他子程序
>
> 在一个线程中执行，协程的切换没有线程切换的开销
>
> 不需要锁机制
>
> 这个例子里面可以看到生产者消费者是同步执行的，并且没有经过锁机制

- 补充生成器知识点
- `n = yield r`
> 返回 `r` 作为 `send()` 的结果
>
> 接收 `send()` 的参数作为 `n`
- `c.send(value)`
> 启动生成器
>
> 将 `value` 作为 `yield` 表达式的值
>
> 执行到下一次碰到 `yield`

In [1]:
import time
# 这是一个Generator(包含了 yield 函数)
from _collections_abc import Iterable
def consumer():
    r = ''
    while True:
        n = yield r
        if not n:
            return
        print('[CONSUMER] Consuming %s...' % n)
        r = '200 OK'
print(isinstance(consumer(), Iterable))
def produce(c):
    c.send(None)
    n = 0
    while n < 5:
        n = n + 1
        print('[PRODUCER] Producing %s...' % n)
        r = c.send(n)
        print('[PRODUCER] Consumer return: %s' % r)
    c.close()

c = consumer()# 返回生成器对象
produce(c)


True
[PRODUCER] Producing 1...
[CONSUMER] Consuming 1...
[PRODUCER] Consumer return: 200 OK
[PRODUCER] Producing 2...
[CONSUMER] Consuming 2...
[PRODUCER] Consumer return: 200 OK
[PRODUCER] Producing 3...
[CONSUMER] Consuming 3...
[PRODUCER] Consumer return: 200 OK
[PRODUCER] Producing 4...
[CONSUMER] Consuming 4...
[PRODUCER] Consumer return: 200 OK
[PRODUCER] Producing 5...
[CONSUMER] Consuming 5...
[PRODUCER] Consumer return: 200 OK


## 使用 `asyncio`
> `@asyncio.coroutine` 可以把 `generator` 标记成 `coroutine`
>
> `coroutine` 内部用 `yield from` 调用另一个 `coroutine` 实现异步操作
>
> `async def` 表明这是一个可以暂停执行的函数
>
> `await ...` 表明执行到这里，相当于 `yield`
>
> `result = await something`    # 暂停在这里,把控制权交给事件循环,等 something 完成
>
> async def 声明的函数只能用 await 来使用
>
> await 只能写在 async def 里面，不能写在普通的 def 里面

- `await` 与 `yield`的区别
- 前者随事件循环自动唤醒（等待的同步事件完成），后者需要调用 send 或者 next
- `await` 相当于没有生成功能，只能暂停时间的 `yield`

In [14]:
import asyncio
import time

async def hello():
    print("Hello world!")
    # 异步调用asyncio.sleep(1):
    await asyncio.sleep(2)
    print("Hello again!")
async def hello2():
    print("Hello world!")
    # 异步调用asyncio.sleep(1):
    await asyncio.sleep(4)
    print("Hello again!")

# asyncio.run(hello()) jupyter 内部已经有一个事件在 run 了，所以不能新创建事件了
# 方法一:两个 hello 会顺序执行
time1 = time.strftime("%X")
await hello()
await hello2()
time2 = time.strftime("%X")
print(time1, time2)

# 方法二:两个 hello 会并发执行，因为被包装生 Task 并交给了事件循环调度
task1 = asyncio.create_task(hello())   # 交给已有的事件循环,后台运行
task2 = asyncio.create_task(hello2())
time3 = time.strftime("%X")
await task1                           # 需要等结果时再 await
await task2
time4 = time.strftime("%X")
print(time3, time4)

# 方法三，方法二的等价写法
print(time.strftime("%X"))
await asyncio.gather(hello(),hello2())
print(time.strftime("%X"))

Hello world!
Hello again!
Hello world!
Hello again!
01:34:36 01:34:41
Hello world!
Hello world!
Hello again!
Hello again!
01:34:41 01:34:45
01:34:45
Hello world!
Hello world!
Hello again!
Hello again!
01:34:49


- 尝试改写之前的生产者消费者模型

In [7]:
import asyncio

async def consumer():
    r = ''
    while True:
        n = yield r                 # 和原来一样:产出 r,等待 asend 传入值
        if not n:
            return
        print('[CONSUMER] Consuming %s...' % n)
        await asyncio.sleep(0.1)    # 模拟耗时的 IO 操作
        r = '200 OK'

async def produce(c):
    await c.asend(None)             # 预激,对应原来的 c.send(None)
    n = 0
    while n < 5:
        n = n + 1
        print('[PRODUCER] Producing %s...' % n)
        r = await c.asend(n)        # 对应原来的 c.send(n)
        print('[PRODUCER] Consumer return: %s' % r)
    await c.aclose()                # 对应原来的 c.close()

async def main():
    c = consumer()
    await produce(c)

await main()

[PRODUCER] Producing 1...
[CONSUMER] Consuming 1...
[PRODUCER] Consumer return: 200 OK
[PRODUCER] Producing 2...
[CONSUMER] Consuming 2...
[PRODUCER] Consumer return: 200 OK
[PRODUCER] Producing 3...
[CONSUMER] Consuming 3...
[PRODUCER] Consumer return: 200 OK
[PRODUCER] Producing 4...
[CONSUMER] Consuming 4...
[PRODUCER] Consumer return: 200 OK
[PRODUCER] Producing 5...
[CONSUMER] Consuming 5...
[PRODUCER] Consumer return: 200 OK


In [9]:
import asyncio

async def wget(host):
    print(f"wget {host}...")
    # 连接80端口:
    reader, writer = await asyncio.open_connection(host, 80)
    # 发送HTTP请求:
    header = f"GET / HTTP/1.0\r\nHost: {host}\r\n\r\n"
    writer.write(header.encode("utf-8"))
    await writer.drain()

    # 读取HTTP响应:
    while True:
        line = await reader.readline()
        if line == b"\r\n":
            break
        print("%s header > %s" % (host, line.decode("utf-8").rstrip()))
    # Ignore the body, close the socket
    writer.close()
    await writer.wait_closed()
    print(f"Done {host}.")

async def main():
    await asyncio.gather(wget("www.sina.com.cn"), wget("www.sohu.com"), wget("www.163.com"))

# asyncio.run(main())
await main()

wget www.sina.com.cn...
wget www.sohu.com...
wget www.163.com...
www.sohu.com header > HTTP/1.1 302 Found
www.sohu.com header > Location: https://www.sohu.com/
www.sohu.com header > Content-Length: 0
www.sohu.com header > X-NWS-LOG-UUID: 14524772234593284090
www.sohu.com header > Connection: close
www.sohu.com header > Server: D0
www.sohu.com header > Date: Wed, 23 Sep 2026 17:16:30 GMT
www.sohu.com header > X-Cache-Lookup: Return Directly
Done www.sohu.com.
www.sina.com.cn header > HTTP/1.1 302 Found
www.sina.com.cn header > Server: Tengine
www.sina.com.cn header > Date: Wed, 23 Sep 2026 17:16:30 GMT
www.sina.com.cn header > Content-Type: text/html
www.sina.com.cn header > Content-Length: 242
www.sina.com.cn header > Connection: close
www.sina.com.cn header > Location: https://www.sina.com.cn/
www.sina.com.cn header > X-DSL-CHECK: 5
www.sina.com.cn header > X-Via-CDN: f=aliyun,s=ens-cache6.cn2917-3,c=119.4.177.3;
www.sina.com.cn header > Via: ens-cache6.cn2917-3[,0]
www.sina.com.cn he